# 03. multi-agent: Handoffs & Router — State machines and parallel routing

## Learning Objectives

- Handoffs pattern: Implements state variable-based dynamic configuration (prompt + tool replacement)
- Trigger a state transition from tool to the `Command` object.
- Router pattern: structured output classification → `Send` API parallel execution → result synthesis

---
# Part A — Handoffs: Customer Support State Machine
---

## 3.1 Environment Setup

This notebook covers two multi-agent patterns:
- **Part A — Handoffs**: State machine pattern where a single agent dynamically swaps prompts and tool based on state variables.
- **Part B — Router**: A pattern that classifies queries, routes them in parallel to professional agents, and synthesizes the results.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

model = ChatOpenAI(model="gpt-4.1")

In [ ]:
# Observability settings (optional) - LangSmith or Langfuse
# Set the key in .env, or uncomment it below and enter it yourself.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: Automatically enabled when LANGSMITH_TRACING=true (no code modification required)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: Pass config={"callbacks": [langfuse_handler]} when calling invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 3.2 Handoffs Overview

The Handoffs pattern is an architecture where a **single agent** dynamically changes its behavior based on state variables. Rather than switching between multiple agents, one agent uses different sets of system prompts and tool depending on the step.

![Handoffs State Machine — Customer Identification → Diagnosis → Resolution → Shutdown](../assets/images/handoffs_state_machine.png)

### Core Mechanism

| mechanism | Description |
|---------|------|
| `current_step` | State variable that tracks the current step. This value determines the agent's behavior |
| `Command(update={...})` | tool returns, triggering a state transition. `current_step` Change + Save Additional Data |
| `@wrap_model_call` | Middleware reads `current_step` and dynamically replaces tool with system prompt |

### Key characteristics of Handoffs

- **State-driven behavior**: Settings are adjusted based on tracked state variables
- **tool-based transitions**: tool returns a `Command` object to update state
- **Direct user interaction**: Messages are processed independently at each stage
- **Persistent state**: The state persists beyond conversation turns

### Important implementation details

When tool updates a message through `Command`, it must include `ToolMessage` with a matching `tool_call_id`. LLM expects responses to be paired with tool calling, so missing them will result in a malformed conversation history.

### When to use

This is ideal when you need sequential constraints, interact directly with the user in each state, or collect information in a specific order in a multi-step flow (e.g. customer support).

## 3.3 SupportState definition

Inherit from `AgentState` and add the `current_step` field. This field determines the current node of the state machine and limits the valid steps to type `Literal`.

The default is `"identify_customer"`, which means all conversations start at the customer identification stage. Afterwards, when tool returns `Command(update={"current_step": "..."})`, it automatically transitions to Next Steps.

In [ ]:
from langchain.agents import AgentState
from typing import Literal

class SupportState(AgentState):
    current_step: Literal[
        "identify_customer", "diagnose_issue",
        "resolve_issue", "close_ticket",
    ] = "identify_customer"

## 3.4 Step-by-step tool definition

Each step is assigned tool that matches the role of that step. tool is divided into two types:

- **State transition tool**: Returns `Command(update={...})` to change `current_step` and store additional data in the state. The string result to be shown to LLM is also delivered to the `result` field.
- **General tool**: Returns a string and does not change state. Used for information retrieval, etc.

Design recommendations:
- State transitions must only occur through tool, which returns `Command`
- Backward transitions (going back to the previous step) are also allowed when necessary.
- Prevent step skipping by validating invalid transitions in middleware

In [ ]:
from langchain_core.tools import tool
from langgraph.types import Command

# --- Identify Customer ---
@tool
def lookup_customer(email: str) -> Command:
    """Track customers by email."""
    return Command(
        update={"customer": {"name": "Alice", "id": "C-1234"}, "current_step": "diagnose_issue"},
        result="Customer found: Alice (C-1234). Go to the diagnosis step.",
    )

In [ ]:
# --- Diagnose Issue ---
@tool
def check_service_status(service_name: str) -> str:
    """Check the current status of the service."""
    return f"서비스 '{service_name}': 정상 (99.9% 가동률)"

In [ ]:
@tool
def escalate_to_resolve(diagnosis: str) -> Command:
    """After diagnosis, move on to resolution steps."""
    return Command(
        update={"diagnosis": diagnosis, "current_step": "resolve_issue"},
        result=f"진단 완료: {diagnosis}. 해결 단계로 이동합니다.",
    )

In [ ]:
# --- Resolve Issue ---
@tool
def apply_fix(fix_type: str, customer_id: str) -> Command:
    """Apply modifications to customer accounts."""
    return Command(
        update={"resolution": {"type": fix_type}},
        result=f"수정 적용됨: {customer_id}에 {fix_type}",
    )

In [ ]:
@tool
def mark_resolved(summary: str) -> Command:
    """Mark the issue as resolved and move it to the Close stage."""
    return Command(
        update={"current_step": "close_ticket", "resolution_summary": summary},
        result="Solved. Go to the termination step.",
    )

In [ ]:
# --- Close Ticket ---
@tool
def send_satisfaction_survey(customer_id: str) -> str:
    """Send a satisfaction survey."""
    return "Survey completed."

@tool
def close_ticket(ticket_id: str, notes: str) -> str:
    """Close the support ticket."""
    return f"티켓 {ticket_id} 종료됨."

## 3.5 @wrap_model_call middleware

`@wrap_model_call` Middleware is the core of the Handoffs pattern. Intercepts LLM calls and dynamically replaces system prompts and available tool depending on `current_step`.

**Sequence of operations:**
1. The middleware reads the `current_step` value from the state
2. Look up the settings (prompt + tool) for that step in the `STEP_CONFIG` dictionary.
3. Override `config` and pass it to LLM
4. Call LLM with modified settings with `next_fn(state, config)`

This is the core mechanic of Handoffs: a single agent will have completely different personas and abilities depending on their status. Achieve dynamic behavior changes with a single middleware, without the need to create multiple agents.

In [ ]:
STEP_CONFIG = {
    "identify_customer": {
        "tools": [lookup_customer],
        "system_prompt": "Identify your customers. Request your email or account ID.",
    },
    "diagnose_issue": {
        "tools": [check_service_status, escalate_to_resolve],
        "system_prompt": "Diagnose the issue. Call escalate_to_resolve after using tool.",
    },
}

In [ ]:
STEP_CONFIG["resolve_issue"] = {
    "tools": [apply_fix, mark_resolved],
    "system_prompt": "Solve the issue. After applying the fix, call mark_resolved.",
}
STEP_CONFIG["close_ticket"] = {
    "tools": [send_satisfaction_survey, close_ticket],
    "system_prompt": "Thank your customers, send surveys, and close tickets.",
}

In [ ]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def step_middleware(request, handler):
    """Dynamically configures agents based on current_step."""
    step = request.state.get("current_step", "identify_customer")
    cfg = STEP_CONFIG[step]
    request = request.override(
        system_prompt=cfg["system_prompt"],
        tools=cfg["tools"],
    )
    return handler(request)

## 3.6 Agent creation and execution flow

When creating an agent, register all tool, but specify `state_schema=SupportState` and middleware. Because the middleware filters tool according to `current_step` at runtime, each step only exposes tool for that step to the LLM.

**Example execution flow:**
This is done automatically by tool, which returns ```
[identify_customer] User: "로그인이 안 돼요. 이메일: alice@example.com"
  → lookup_customer("alice@example.com")
  ← Command(update={customer: {...}, current_step: "diagnose_issue"})

[diagnose_issue] Agent: "계정을 찾았습니다. 어떤 문제가 있나요?"
  → check_service_status("auth-service") → "healthy"
  → escalate_to_resolve("3회 로그인 실패로 잠김")

[resolve_issue] → apply_fix("reset_password", "C-1234")
  → mark_resolved("비밀번호 재설정 완료")

[close_ticket] → send_satisfaction_survey("C-1234")
  → close_ticket("T-5678", notes="비밀번호 재설정")
```

각 단계 전이는 `Command`.

In [ ]:
from langchain.agents import create_agent

all_tools = [
    lookup_customer, check_service_status,
    escalate_to_resolve, apply_fix, mark_resolved,
    send_satisfaction_survey, close_ticket,
]
support_agent = create_agent(
    model="gpt-4.1", tools=all_tools,
    state_schema=SupportState, middleware=[step_middleware],
)

In [ ]:
# [identify_customer]
#   User: "Can't log in. Email: alice@example.com"
#   Agent -> lookup_customer("alice@example.com")
#        <- Command(update={current_step: "diagnose_issue"})
#
# [diagnose_issue] (auto transition)
#   Agent -> check_service_status("auth-service")
#   Agent -> escalate_to_resolve("Locked out")
#
# [resolve_issue] -> [close_ticket]

---
# Part B — Router: Parallel routing and result synthesis
---

## 3.7 Router Overview

The Router pattern is an architecture that classifies input and routes it to specialized agents. Unlike the Subagents pattern, Router distributes queries via a dedicated classification step (either a single LLM call or rule-based logic).

![Router fanout/fanin pattern — Router→Parallel Agent→Reducer](../assets/images/router_fanout_fanin.png)

### Pipeline

| steps | Role | implementation |
|------|------|------|
| **Classification** | Analyze queries to create related sources and subqueries | `with_structured_output(QueryClassification)` |
| **Parallel Dispatch** | Deliver subqueries to each classified source simultaneously | `Send` API |
| **Result synthesis (Reduction)** | Gather all agent results to create a unified response | Reducer node + LLM |

### Router vs. Subagents Comparison

Routers have a “dedicated routing stage (classification),” while Subagents have “supervisor agents dynamically” deciding what to call. Router is suitable when distinct knowledge domains (verticals) are clearly distinguished and parallel queries are required.

### Architecture Mode

- **Stateless**: Each request is routed independently (no memory)
- **Stateful**: Supports multi-turn interactions by maintaining conversation history. An alternative is to wrap a stateless router with tool, or have the router itself manage the state directly.

## 3.8 RouterState and classification schema

`QueryClassification` is a Pydantic model, which classifies queries into a structured form through LLM's `with_structured_output()`. `RouterState` tracks classification results, source lists, subqueries, and agent results.

Key fields in the classification schema:
- `sources`: Which knowledge sources are relevant (multiple choices possible)
- `reasoning`: Explain why you selected the source
- `sub_queries`: Subquery optimized for each source (original query reorganized for each source)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class SubQuery(BaseModel):
    """Subqueries by source."""
    source: Literal["github", "notion", "slack"] = Field(description="Knowledge source.")
    query: str = Field(description="Search queries optimized for that source.")

class QueryClassification(BaseModel):
    """Classification results from user queries."""
    sources: list[Literal["github", "notion", "slack"]] = Field(
        description="Related knowledge sources."
    )
    reasoning: str = Field(description="Why you chose that source.")
    sub_queries: list[SubQuery] = Field(description="Subqueries by source.")

In [ ]:
from langchain.agents import AgentState

class RouterState(AgentState):
    classification: QueryClassification = None
    sources: list[str] = []
    sub_queries: list[SubQuery] = []
    agent_results: list[dict] = []

## 3.9 Classification Node

`with_structured_output` classifies queries by source and creates subqueries optimized for each source.

### Classification example

| user query | Category Source | Reason |
|-----------|----------|------|
| "How to deploy the auth service" | `["github", "notion"]` | Distribution code exists on GitHub, procedure documentation exists on Notion |
| "How the decision was made to change the API" | `["slack", "notion"]` | Discussions are recorded in Slack, decision documents are recorded in Notion |
| “Login bug PR” | `["github"]` | PR only exists on GitHub |
| “Onboarding Process and Starter Repo” | `["github", "notion", "slack"]` | Repo is GitHub, process is Notion, context is Slack |

Subquery creation is important: optimize the original query “auth service deployment” to `"auth service deployment scripts CI/CD pipeline"` for GitHub and `"auth service deployment process procedure runbook"` for Notion, respectively.

In [ ]:
from langchain_openai import ChatOpenAI

classifier = ChatOpenAI(model="gpt-4.1").with_structured_output(
    QueryClassification
)

def route_query(state):
    """Classify queries and make routing decisions."""
    msg = state["messages"][-1].content
    cls = classifier.invoke(f"분류하고 하위 쿼리를 생성하세요:\n\n{msg}", config=lf_config)
    sq_dict = {sq.source: sq.query for sq in cls.sub_queries}
    return {"classification": cls, "sources": cls.sources, "sub_queries": cls.sub_queries}

## 3.10 Parallel Routing (Send API)

The `Send` API dispatches subqueries to each classified source simultaneously. In the form of `Send(node_name, payload)`, data is passed in parallel to specific nodes in the graph.

This parallel execution is the core strength of the Router pattern: sequentially querying multiple knowledge sources adds up the latency, but parallel execution via `Send` takes only as long as the response time of the slowest source.

### Add new source

The Router pattern is simple to extend:
1. Define source-specific tool
2. Create a professional agent
3. Add new source to `QueryClassification.sources`
4. Add agent nodes to the graph
5. Connect to Reducer

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def search_github_code(query: str) -> str:
    """Search the GitHub repository."""
    return f"'{query}'에 대한 GitHub 결과"

@tool
def search_notion_pages(query: str) -> str:
    """Search for your Notion workspace."""
    return f"'{query}'에 대한 Notion 결과"

In [ ]:
@tool
def search_slack_messages(query: str) -> str:
    """Search for Slack messages."""
    return f"'{query}'에 대한 Slack 결과"

In [ ]:
github_agent = create_agent(
    model="gpt-4.1", tools=[search_github_code],
    system_prompt="Search for code and PRs on GitHub.",
    name="github_agent",
)
notion_agent = create_agent(
    model="gpt-4.1", tools=[search_notion_pages],
    system_prompt="Search documents in Notion.",
    name="notion_agent",
)

In [ ]:
slack_agent = create_agent(
    model="gpt-4.1", tools=[search_slack_messages],
    system_prompt="Search for discussions in Slack.",
    name="slack_agent",
)

In [ ]:
from langgraph.types import Send

def dispatch_to_agents(state):
    """Subqueries are passed to agents in parallel."""
    cls = state["classification"]
    sq_dict = {sq.source: sq.query for sq in cls.sub_queries}
    return [
        Send(src, {"messages": [{"role": "user", "content": sq_dict.get(src, "")}], "source": src})
        for src in cls.sources
    ]

## 3.11 Result synthesis

Reducer collects the results from all agents and uses LLM to synthesize an integrated response. When synthesizing, the source of each information is cited so that the user can determine where the information came from.

The synthesis prompt instructs you to indicate the source. For example, respond with "The deployment script is in the `payment-service` repo on GitHub (GitHub), and for deployment procedures, please refer to Notion's 'Payment Service Ops' document (Notion)."

In [ ]:
def reduce_results(state):
    """Aggregates results from all agents."""
    results = state.get("agent_results", [])
    formatted = "\n".join(f"[{r['source']}] {r['content']}" for r in results)
    prompt = f"일관된 답변으로 합성하세요. 출처를 인용하세요.\n\n{formatted}"
    resp = ChatOpenAI(model="gpt-4.1").invoke(prompt, config=lf_config)
    return {"messages": [{"role": "assistant", "content": resp.content}]}

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(RouterState)
graph.add_node("router", route_query)
graph.add_node("github", github_agent)
graph.add_node("notion", notion_agent)
graph.add_node("slack", slack_agent)
graph.add_node("reducer", reduce_results)

In [ ]:
graph.add_edge(START, "router")
graph.add_conditional_edges("router", dispatch_to_agents)
graph.add_edge("github", "reducer")
graph.add_edge("notion", "reducer")
graph.add_edge("slack", "reducer")
graph.add_edge("reducer", END)

app = graph.compile()

In [ ]:
response = app.invoke({
    "messages": [{"role": "user",
        "content": "How do I deploy my payment service?"}]
}, config=lf_config)
print(response["messages"][-1].content)

## Summary

### Part A — Handoffs

| Item | core |
|------|------|
| **Pattern** | Single agent + `current_step` based dynamic configuration |
| **Transition** | `Command(update={"current_step": "next"})` |
| **Dynamic Configuration** | prompt with `@wrap_model_call` + replace tool |

### Part B — Router

| Item | core |
|------|------|
| **Category** | `with_structured_output(QueryClassification)` |
| **Parallel** | `Send(source, payload)` API |
| **Synthesis** | LLM integrated response from reducer node |

### Next Steps
→ **[04_context_memory.ipynb](./04_context_memory.ipynb)**: Learn context engineering and memory.